# 02 — SQL Exploration & Data Preparation

## Goal
Build the final aggregated table (vendor_sales_summary) from 6 raw tables.

## Steps Completed
1. Connected to InventoryDB
2. Explored all 6 tables
3. Built freight, purchase and sales summaries
4. Merged into one final table (11,101 rows)
5. Cleaned missing values, fixed data types
6. Created new features: GrossProfit, ProfitMargin, StockTurnover, SalesToPurchaseRatio
7. Filtered to 8,891 meaningful records
8. Saved vendor_sales_summary back to database

In [2]:
import pandas as pd
import urllib
from sqlalchemy import create_engine

# Database Connection
params = urllib.parse.quote_plus(
    'DRIVER={ODBC Driver 17 for SQL Server};'
    'SERVER=Wasif\\SQLEXPRESS;'
    'DATABASE=InventoryDB;'
    'Trusted_Connection=yes;'
    'TrustServerCertificate=yes;'
)
engine = create_engine(
    f'mssql+pyodbc:///?odbc_connect={params}',
    fast_executemany=True
)

print("✅ Connected!")

✅ Connected!


In [3]:
# Check all tables and their row counts
query = """
SELECT 
    t.name AS TableName,
    SUM(p.rows) AS TotalRows
FROM sys.tables t
JOIN sys.partitions p 
    ON t.object_id = p.object_id
WHERE p.index_id IN (0,1)
GROUP BY t.name
ORDER BY TotalRows DESC
"""

df_tables = pd.read_sql(query, engine)
print(df_tables)

         TableName  TotalRows
0            sales   12825363
1        purchases    2372474
2    end_inventory     224489
3  begin_inventory     206529
4  purchase_prices      12261
5   vendor_invoice       5543


In [4]:
# Look at top 5 rows of each table
tables = ['vendor_invoice', 'purchase_prices', 
          'purchases', 'sales', 
          'begin_inventory', 'end_inventory']

for table in tables:
    print(f"\n{'='*50}")
    print(f"TABLE: {table}")
    print('='*50)
    df = pd.read_sql(f"SELECT TOP 5 * FROM {table}", engine)
    print(df.to_string())


TABLE: vendor_invoice
   VendorNumber                   VendorName InvoiceDate  PONumber      PODate     PayDate  Quantity    Dollars  Freight Approval
0           105  ALTAMAR BRANDS LLC           2024-01-04      8124  2023-12-21  2024-02-16         6     214.26     3.47     None
1          4466  AMERICAN VINTAGE BEVERAGE    2024-01-07      8137  2023-12-22  2024-02-21        15     140.55     8.57     None
2           388  ATLANTIC IMPORTING COMPANY   2024-01-09      8169  2023-12-24  2024-02-16         5     106.60     4.61     None
3           480  BACARDI USA INC              2024-01-12      8106  2023-12-20  2024-02-05     10100  137483.78  2935.20     None
4           516  BANFI PRODUCTS CORP          2024-01-07      8170  2023-12-24  2024-02-12      1935   15527.25   429.20     None

TABLE: purchase_prices
   Brand                  Description  Price   Size Volume  Classification  PurchasePrice  VendorNumber                   VendorName
0     58  Gekkeikan Black & Gold Sake  1

In [5]:
# Check sales and purchases columns
print("PURCHASES COLUMNS:")
df_p = pd.read_sql("SELECT TOP 1 * FROM purchases", engine)
for col in df_p.columns:
    print(f"  - {col}")

print("\nSALES COLUMNS:")
df_s = pd.read_sql("SELECT TOP 1 * FROM sales", engine)
for col in df_s.columns:
    print(f"  - {col}")

PURCHASES COLUMNS:
  - InventoryId
  - Store
  - Brand
  - Description
  - Size
  - VendorNumber
  - VendorName
  - PONumber
  - PODate
  - ReceivingDate
  - InvoiceDate
  - PayDate
  - PurchasePrice
  - Quantity
  - Dollars
  - Classification

SALES COLUMNS:
  - InventoryId
  - Store
  - Brand
  - Description
  - Size
  - SalesQuantity
  - SalesDollars
  - SalesPrice
  - SalesDate
  - Volume
  - Classification
  - ExciseTax
  - VendorNo
  - VendorName


In [6]:
# SUMMARY 1 — Freight per vendor
freight_summary = pd.read_sql("""
    SELECT 
        VendorNumber,
        VendorName,
        SUM(Freight) AS FreightCost
    FROM vendor_invoice
    GROUP BY VendorNumber, VendorName
""", engine)

print(f"Freight Summary: {freight_summary.shape}")
print(freight_summary.head())

Freight Summary: (128, 3)
   VendorNumber                       VendorName  FreightCost
0            54      AAPER ALCOHOL & CHEMICAL CO         0.48
1            60      ADAMBA IMPORTS INTL INC           367.52
2          1703  ALISA CARR BEVERAGES                  172.00
3           105      ALTAMAR BRANDS LLC                 62.39
4           200      AMERICAN SPIRITS EXCHANGE           6.19


In [7]:
# SUMMARY 2 — Purchase summary per vendor and brand
purchase_summary = pd.read_sql("""
    SELECT 
        p.VendorNumber,
        p.VendorName,
        p.Brand,
        p.Description,
        pp.Volume,
        pp.Price AS ActualPrice,
        SUM(p.PurchasePrice) AS PurchasePrice,
        SUM(p.Quantity) AS TotalPurchaseQuantity,
        SUM(p.Dollars) AS TotalPurchaseDollars
    FROM purchases p
    JOIN purchase_prices pp
        ON p.Brand = pp.Brand
    WHERE p.PurchasePrice > 0
    GROUP BY 
        p.VendorNumber, 
        p.VendorName, 
        p.Brand, 
        p.Description,
        pp.Volume,
        pp.Price
""", engine)

print(f"Purchase Summary: {purchase_summary.shape}")
print(purchase_summary.head())

Purchase Summary: (10692, 9)
   VendorNumber                               VendorName  Brand  \
0             2  IRA GOLDMAN AND WILLIAMS, LLP            90085   
1             2  IRA GOLDMAN AND WILLIAMS, LLP            90609   
2            54              AAPER ALCOHOL & CHEMICAL CO    990   
3            60              ADAMBA IMPORTS INTL INC        771   
4            60              ADAMBA IMPORTS INTL INC       3401   

                    Description Volume  ActualPrice  PurchasePrice  \
0  Ch Lilian 09 Ladouys St Este    750        36.99          71.58   
1  Flavor Essence Variety 5 Pak  162.5        24.99         170.00   
2       Ethyl Alcohol 200 Proof   3750       134.49         105.07   
3   Bak's Krupnik Honey Liqueur    750        14.99          45.76   
4                  Vesica Vodka   1750        14.99          11.10   

   TotalPurchaseQuantity  TotalPurchaseDollars  
0                      8                190.88  
1                    320               5440.00  


In [10]:
# SUMMARY 3 — Sales summary per vendor and brand
sales_summary = pd.read_sql("""
    SELECT 
        VendorNo AS VendorNumber,
        VendorName,
        Brand,
        SUM(SalesQuantity) AS TotalSalesQuantity,
        SUM(SalesDollars) AS TotalSalesDollars,
        SUM(SalesPrice) AS TotalSalesPrice,
        SUM(ExciseTax) AS TotalExciseTax
    FROM sales
    GROUP BY 
        VendorNo,
        VendorName,
        Brand
""", engine)

print(f"Sales Summary: {sales_summary.shape}")
print(sales_summary.head())

Sales Summary: (11272, 7)
   VendorNumber                               VendorName  Brand  \
0             2  IRA GOLDMAN AND WILLIAMS, LLP            90085   
1             2  IRA GOLDMAN AND WILLIAMS, LLP            90609   
2            60              ADAMBA IMPORTS INTL INC        771   
3            60              ADAMBA IMPORTS INTL INC       3979   
4           105              ALTAMAR BRANDS LLC            2529   

   TotalSalesQuantity  TotalSalesDollars  TotalSalesPrice  TotalExciseTax  
0                  18             665.82           295.92            2.00  
1                  24             599.76           449.82            0.52  
2                  47             704.53           494.67           37.01  
3                3931           66871.69         41682.51         7224.06  
4                  12             359.88            59.98            9.44  


In [11]:
# Merge all 3 summaries into final table
vendor_sales_summary = purchase_summary.merge(
    sales_summary[['VendorNumber', 'Brand', 'TotalSalesQuantity', 
                   'TotalSalesDollars', 'TotalSalesPrice', 'TotalExciseTax']],
    on=['VendorNumber', 'Brand'],
    how='left'
).merge(
    freight_summary[['VendorNumber', 'FreightCost']],
    on='VendorNumber',
    how='left'
)

print(f"Final Table Shape: {vendor_sales_summary.shape}")
print(vendor_sales_summary.head())

Final Table Shape: (11101, 14)
   VendorNumber                               VendorName  Brand  \
0             2  IRA GOLDMAN AND WILLIAMS, LLP            90085   
1             2  IRA GOLDMAN AND WILLIAMS, LLP            90609   
2            54              AAPER ALCOHOL & CHEMICAL CO    990   
3            60              ADAMBA IMPORTS INTL INC        771   
4            60              ADAMBA IMPORTS INTL INC       3401   

                    Description Volume  ActualPrice  PurchasePrice  \
0  Ch Lilian 09 Ladouys St Este    750        36.99          71.58   
1  Flavor Essence Variety 5 Pak  162.5        24.99         170.00   
2       Ethyl Alcohol 200 Proof   3750       134.49         105.07   
3   Bak's Krupnik Honey Liqueur    750        14.99          45.76   
4                  Vesica Vodka   1750        14.99          11.10   

   TotalPurchaseQuantity  TotalPurchaseDollars  TotalSalesQuantity  \
0                      8                190.88                18.0   
1    

In [12]:
# Step 1 — Check missing values
print("Missing Values:")
print(vendor_sales_summary.isnull().sum())

Missing Values:
VendorNumber               0
VendorName                 0
Brand                      0
Description                0
Volume                     0
ActualPrice                0
PurchasePrice              0
TotalPurchaseQuantity      0
TotalPurchaseDollars       0
TotalSalesQuantity       184
TotalSalesDollars        184
TotalSalesPrice          184
TotalExciseTax           184
FreightCost                0
dtype: int64


In [13]:
# Step 2 — Fix missing values
# Fill sales columns with 0 (products purchased but not sold)
vendor_sales_summary[['TotalSalesQuantity', 'TotalSalesDollars', 
                       'TotalSalesPrice', 'TotalExciseTax']] = \
vendor_sales_summary[['TotalSalesQuantity', 'TotalSalesDollars', 
                       'TotalSalesPrice', 'TotalExciseTax']].fillna(0)

# Step 3 — Fix VendorName whitespace
vendor_sales_summary['VendorName'] = vendor_sales_summary['VendorName'].str.strip()

# Step 4 — Fix Volume column to float
vendor_sales_summary['Volume'] = vendor_sales_summary['Volume'].astype(float)

# Verify
print("Missing values after cleaning:")
print(vendor_sales_summary.isnull().sum())
print(f"\nShape: {vendor_sales_summary.shape}")
print(f"\nData Types:")
print(vendor_sales_summary.dtypes)

Missing values after cleaning:
VendorNumber             0
VendorName               0
Brand                    0
Description              0
Volume                   0
ActualPrice              0
PurchasePrice            0
TotalPurchaseQuantity    0
TotalPurchaseDollars     0
TotalSalesQuantity       0
TotalSalesDollars        0
TotalSalesPrice          0
TotalExciseTax           0
FreightCost              0
dtype: int64

Shape: (11101, 14)

Data Types:
VendorNumber               int64
VendorName                   str
Brand                      int64
Description                  str
Volume                   float64
ActualPrice              float64
PurchasePrice            float64
TotalPurchaseQuantity      int64
TotalPurchaseDollars     float64
TotalSalesQuantity       float64
TotalSalesDollars        float64
TotalSalesPrice          float64
TotalExciseTax           float64
FreightCost              float64
dtype: object


In [14]:
# Step 5 — Create new features

# Gross Profit = Sales - Purchase
vendor_sales_summary['GrossProfit'] = (
    vendor_sales_summary['TotalSalesDollars'] - 
    vendor_sales_summary['TotalPurchaseDollars']
)

# Profit Margin % 
vendor_sales_summary['ProfitMargin'] = (
    vendor_sales_summary['GrossProfit'] / 
    vendor_sales_summary['TotalSalesDollars'] * 100
)

# Stock Turnover = Sold / Purchased
vendor_sales_summary['StockTurnover'] = (
    vendor_sales_summary['TotalSalesQuantity'] / 
    vendor_sales_summary['TotalPurchaseQuantity']
)

# Sales to Purchase Ratio
vendor_sales_summary['SalesToPurchaseRatio'] = (
    vendor_sales_summary['TotalSalesDollars'] / 
    vendor_sales_summary['TotalPurchaseDollars']
)

print("New columns added:")
print(vendor_sales_summary[['GrossProfit', 'ProfitMargin', 
                             'StockTurnover', 
                             'SalesToPurchaseRatio']].head())
print(f"\nFinal Shape: {vendor_sales_summary.shape}")

New columns added:
   GrossProfit  ProfitMargin  StockTurnover  SalesToPurchaseRatio
0       474.94     71.331591       2.250000              3.488160
1     -4840.24   -807.029478       0.075000              0.110250
2      -105.07          -inf       0.000000              0.000000
3       258.37     36.672675       1.205128              1.579097
4       -66.60          -inf       0.000000              0.000000

Final Shape: (11101, 18)


In [15]:
# Step 6 — Replace -inf and inf with 0
import numpy as np

vendor_sales_summary.replace([np.inf, -np.inf], 0, inplace=True)

# Step 7 — Filter only meaningful records
vendor_sales_summary = vendor_sales_summary[
    (vendor_sales_summary['GrossProfit'] > 0) &
    (vendor_sales_summary['ProfitMargin'] > 0) &
    (vendor_sales_summary['TotalSalesQuantity'] > 0)
]

print(f"Clean records: {vendor_sales_summary.shape}")
print(vendor_sales_summary[['GrossProfit', 'ProfitMargin', 
                             'StockTurnover']].describe())

Clean records: (8891, 18)
        GrossProfit  ProfitMargin  StockTurnover
count  8.891000e+03  8.891000e+03    8891.000000
mean   1.539059e+04  3.900266e+01       2.051811
std    5.019460e+04  2.148496e+01       6.646421
min    2.842171e-14  1.367480e-14       0.428571
25%    4.933650e+02  2.613190e+01       0.945433
50%    3.066780e+03  3.365098e+01       1.000000
75%    1.226432e+04  4.445663e+01       1.130267
max    1.290668e+06  9.971666e+01     274.500000


In [16]:
# Step 8 — Save final table to database
print("Saving vendor_sales_summary to database...")

vendor_sales_summary.to_sql(
    name='vendor_sales_summary',
    con=engine,
    if_exists='replace',
    index=False,
    chunksize=1000
)

print("✅ Table saved successfully!")

# Verify
df_check = pd.read_sql(
    "SELECT COUNT(*) AS TotalRows FROM vendor_sales_summary", 
    engine
)
print(f"Rows in database: {df_check['TotalRows'][0]:,}")

Saving vendor_sales_summary to database...
✅ Table saved successfully!
Rows in database: 8,891


In [17]:
import logging
import os

os.makedirs('../logs', exist_ok=True)

# Create a new log file for this notebook
logger = logging.getLogger('sql_exploration')
logger.setLevel(logging.DEBUG)

handler = logging.FileHandler('../logs/sql_exploration.log', mode='a')
handler.setFormatter(logging.Formatter('%(asctime)s - %(levelname)s - %(message)s'))
logger.addHandler(handler)

# Log all steps
logger.info('=== SQL Exploration & Data Preparation Started ===')
logger.info('Step 1: Connected to InventoryDB successfully')
logger.info('Step 2: Explored all 6 tables - sales, purchases, vendor_invoice, purchase_prices, begin_inventory, end_inventory')
logger.info('Step 3: Built freight_summary - 128 vendors')
logger.info('Step 4: Built purchase_summary - 10,692 records')
logger.info('Step 5: Built sales_summary - 11,272 records')
logger.info('Step 6: Merged all 3 summaries - 11,101 records, 14 columns')
logger.info('Step 7: Cleaned data - fixed nulls, inf values, whitespace, data types')
logger.info('Step 8: Created new features - GrossProfit, ProfitMargin, StockTurnover, SalesToPurchaseRatio')
logger.info('Step 9: Filtered to meaningful records - 8,891 rows, 18 columns')
logger.info('Step 10: Saved vendor_sales_summary to InventoryDB successfully')
logger.info('=== SQL Exploration & Data Preparation Completed ===')

print("✅ Log updated!")

✅ Log updated!
